# Step 1: Unzip historical ground reflectivity files and store in scratch as .nc files

In [2]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import sys
import metpy
import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import metpy.calc as mpcalc
import pandas as pd
from netCDF4 import Dataset
import pyproj
import os
import glob
from datetime import datetime
import seaborn as sns
import netCDF4
from netCDF4 import Dataset
from metpy.units import units
import dask
import xarray as xr
from shapely import Polygon
import regionmask
import geopandas as gpd
import logging
import dask
from dask.distributed import Client
from dask import delayed
import functools
import zipfile
import tempfile
import wradlib as wrl
sys.path.append(os.path.join(r"/home/563/ac9768/GBR/scripts/Paper_figures")) 

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
barra_towns = xr.open_dataset("/g/data/q90/ac9768/GBR/barra-2/barra-2_850hPa_winds_townsville.nc", engine="h5netcdf",chunks="auto")
barra_cairns = xr.open_dataset("/g/data/q90/ac9768/GBR/barra-2/barra-2_850hPa_winds_cairns.nc", engine="h5netcdf",chunks="auto")
barra_willis = xr.open_dataset("/g/data/q90/ac9768/GBR/barra-2/barra-2_850hPa_winds_willis_island.nc", engine="h5netcdf",chunks="auto")

In [5]:
client = Client(n_workers=28,threads_per_worker=1,memory_limit=None,silence_logs=logging.ERROR,dashboard_address='8787') 
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 28
Total threads: 28,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37945,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:42711,Total threads: 1
Dashboard: /proxy/39569/status,Memory: 0 B
Nanny: tcp://127.0.0.1:33185,


In [13]:
def path_to_radar_ds_zip2nc(radar_site_no: str, temp_dir: str = "/scratch/v46/ac9768/radar_grndref/",F06: bool=False):
    """Create list of file paths for the chosen radar ID.
    Walks the full prcp-crate directory and returns all .zip-contained .nc files.

    Args:
        radar_site_no (str): String of radar ID number; towns = 73, cairns = "19", willis island = "41"
        temp_dir (str): Directory to store .nc files, default = "/scratch/v46/ac9768/radar_grndref/"
    Returns:
        list: List of extracted .nc file path strings
    """
    # analysis years
    year  = ['2012','2013','2014','2015','2016','2017','2018','2019','2020','2021','2022']
    # analysis months
    month = ['01','02','03','04']
    zip_list = sorted(
        f
        for y in year
        for m in month
        for f in glob.glob(f"/g/data/rq0/hist_gndrefl/v2026/{radar_site_no}/{y}/{radar_site_no}_{y}{m}*.zip")
    )

    if not zip_list:
        raise FileNotFoundError(f"No .zip files found for radar site '{radar_site_no}' — check the path.")
        
    tmp_dir = tempfile.mkdtemp(prefix=f"radar_{radar_site_no}_", dir=temp_dir)
    nc_files = []

    # for f06/f11 just need metadata from one file
    if F06:
        with zipfile.ZipFile(zip_list[0], 'r') as zf:
            for member in zf.namelist():
                if member.endswith('.nc'):
                    zf.extract(member, tmp_dir)
                    nc_files.append(os.path.join(tmp_dir, member))
    else:
        for zp in zip_list:
            with zipfile.ZipFile(zp, 'r') as zf:
                for member in zf.namelist():
                    if member.endswith('.nc'):
                        zf.extract(member, tmp_dir)
                        nc_files.append(os.path.join(tmp_dir, member))

    nc_files.sort()
    return nc_files

In [16]:
# to unzip files and convert to nc - only one site done at a time (fills inode allocation otherwise)
# towns_radar_files = path_to_radar_ds_zip2nc("73",F06=True)
cairns_radar_files = path_to_radar_ds_zip2nc("19",F06=True)
# willis_radar_files = path_to_radar_ds_zip2nc("41")